# LCA + CVRP for Energy Distribution Logistics

End-to-end walkthrough of the pipeline on the synthetic network.

**Note.** This notebook runs on generated data with placeholder characterization factors. It demonstrates the method; it does not reproduce the figures reported at YAEM 2026. See `data/README.md` for why.


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt

import config
import generate_synthetic_data
import cvrp_solver
import lca_model


## 1. Build the synthetic network

885 dealers clustered around 13 facilities, with right-skewed demand.


In [ ]:
generate_synthetic_data.main()

dealers = pd.read_csv(config.DEALERS_CSV)
facilities = pd.read_csv(config.FACILITIES_CSV)
dealers.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dealers['longitude'], dealers['latitude'], s=6, alpha=0.45, label='Dealers')
ax.scatter(facilities['longitude'], facilities['latitude'], s=140, marker='s',
           edgecolor='black', linewidth=1.2, label='Facilities', zorder=3)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Synthetic distribution network (no real locations)')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 2. Solve the routing problem

Baseline is a capacitated nearest-neighbour policy. Optimized is OR-Tools with guided local search. The gap between them is the operational lever.


In [ ]:
routes = cvrp_solver.run()
routes


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(routes))
ax.bar([i - 0.2 for i in x], routes['baseline_km'], width=0.4, label='Baseline')
ax.bar([i + 0.2 for i in x], routes['optimized_km'], width=0.4, label='Optimized')
ax.set_xticks(list(x)); ax.set_xticklabels(routes['facility_id'], rotation=45)
ax.set_ylabel('Distance (km)')
ax.set_title('Routing distance per facility, baseline vs optimized')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## 3. Characterize the two pathways

Transport enters the inventory as tonne-kilometres, which is what couples the routing result to the impact result.


In [ ]:
lca_model.main()


## 4. Next steps

Replace every `TODO(nunu)` in `src/config.py` with the values used in the study, then swap the synthetic CSVs for the real network to reproduce the published figures.
